# Lief Data Science Test
---

## Question 1 JSON Data Transformation

Transform the received JSON into a flat results list by resolving all ID references:
- `gender_id` → gender label
- `adore_car` → car object (brand + country)
- `car_brand` list → brand names
- `color` list → full RGB objects
- `user_id` → country label

In [125]:
import json
import pandas as pd

In [126]:
# input data

received = {
    "students": [
        {"user_id": 1, "first_name": "Dorree", "last_name": "Stanlick", "gender_id": "a2", "adore_car": "c4", "car_brand": ["c1", "c4"], "color": ["IndianRed", "LightCoral"]},
        {"user_id": 2, "first_name": "Gert", "last_name": "Fake", "gender_id": "a1", "adore_car": "c1", "car_brand": ["c1", "c2"], "color": ["DarkSalmon", "LightSalmon"]},
        {"user_id": 3, "first_name": "Rene", "last_name": "Dust", "gender_id": "a2", "adore_car": "c5", "car_brand": ["c3", "c5"], "color": ["Salmon", "LightCoral"]},
        {"user_id": 4, "first_name": "Clo", "last_name": "Cordes", "gender_id": "a1", "adore_car": "c4", "car_brand": ["c2", "c4", "c5"], "color": ["DarkSalmon"]},
        {"user_id": 5, "first_name": "Emlynne", "last_name": "Kettley", "gender_id": "a2", "adore_car": "c1", "car_brand": ["c1"], "color": ["IndianRed", "LightSalmon", "LightCoral", "Salmon"]}
    ],
    "gender": [
        {"gender_id": "a1", "label": "male"},
        {"gender_id": "a2", "label": "female"}
    ],
    "cars": [
        {"car_id": "c1", "car_brand": "Subaru", "car_make": "Japan"},
        {"car_id": "c2", "car_brand": "Dodge", "car_make": "United States"},
        {"car_id": "c3", "car_brand": "Volkswagen", "car_make": "Germany"},
        {"car_id": "c4", "car_brand": "Hyundai", "car_make": "Korea"},
        {"car_id": "c5", "car_brand": "Toyota", "car_make": "Japan"}
    ],
    "rgbCode": [
        {"hex": "#CD5C5C", "label": "IndianRed", "rgb": "205, 92, 92"},
        {"hex": "#F08080", "label": "LightCoral", "rgb": "240, 128, 128"},
        {"hex": "#FA8072", "label": "Salmon", "rgb": "250, 128, 114"},
        {"hex": "#E9967A", "label": "DarkSalmon", "rgb": "233, 150, 122"},
        {"hex": "#FFA07A", "label": "LightSalmon", "rgb": "255, 160, 122"}
    ],
    "countries": [
        {"label": "Czech Republic", "user_ids": [{"userId": "1"}, {"userId": "3"}, {"userId": "5"}]},
        {"label": "Colombia", "user_ids": [{"userId": "2"}, {"userId": "4"}]}
    ]
}

In [127]:
# build lookup tables from reference data

df = pd.DataFrame(received["students"])
df_cars = pd.DataFrame(received["cars"])
df_color = pd.DataFrame(received["rgbCode"])

gender_map = {g["gender_id"]: g["label"] for g in received["gender"]}
brand_map = df_cars.set_index("car_id")["car_brand"].to_dict()
make_map = df_cars.set_index("car_id")["car_make"].to_dict()
color_lookup = df_color.set_index("label").to_dict(orient="index")

country_map = {}
for c in received["countries"]:
    for uid in c["user_ids"]:
        country_map[int(uid["userId"])] = c["label"]

df

,user_id,first_name,last_name,gender_id,adore_car,car_brand,color
0,1,Dorree,Stanlick,a2,c4,"[c1, c4]","[IndianRed, LightCoral]"
1,2,Gert,Fake,a1,c1,"[c1, c2]","[DarkSalmon, LightSalmon]"
2,3,Rene,Dust,a2,c5,"[c3, c5]","[Salmon, LightCoral]"
3,4,Clo,Cordes,a1,c4,"[c2, c4, c5]",[DarkSalmon]
4,5,Emlynne,Kettley,a2,c1,[c1],"[IndianRed, LightSalmon, LightCoral, Salmon]"


In [128]:
# resolve all IDs to actual values
# Transform the data
df["gender"] = df["gender_id"].map(gender_map)
df["adore_car"] = df["adore_car"].map(lambda cid: {"car_brand": brand_map[cid], "brand_from": make_map[cid]})
df["car_brand"] = df["car_brand"].apply(lambda ids: [brand_map[cid] for cid in ids])
df["colors"] = df["color"].apply(lambda names: [{"hex": color_lookup[n]["hex"], "label": n, "rgb": color_lookup[n]["rgb"]} for n in names])
df["countries"] = df["user_id"].map(country_map)

df[["user_id", "first_name", "gender", "adore_car", "car_brand", "countries"]]

,user_id,first_name,gender,adore_car,car_brand,countries
0,1,Dorree,female,"{'car_brand': 'Hyundai', 'brand_from': 'Korea'}","[Subaru, Hyundai]",Czech Republic
1,2,Gert,male,"{'car_brand': 'Subaru', 'brand_from': 'Japan'}","[Subaru, Dodge]",Colombia
2,3,Rene,female,"{'car_brand': 'Toyota', 'brand_from': 'Japan'}","[Volkswagen, Toyota]",Czech Republic
3,4,Clo,male,"{'car_brand': 'Hyundai', 'brand_from': 'Korea'}","[Dodge, Hyundai, Toyota]",Colombia
4,5,Emlynne,female,"{'car_brand': 'Subaru', 'brand_from': 'Japan'}",[Subaru],Czech Republic


In [129]:
# select columns and output as JSON

result_df = df[["user_id", "first_name", "last_name", "gender",
                "adore_car", "car_brand", "countries", "colors"]]

results = result_df.to_dict(orient="records")
print(json.dumps(results, indent=2, ensure_ascii=False))

[
  {
    "user_id": 1,
    "first_name": "Dorree",
    "last_name": "Stanlick",
    "gender": "female",
    "adore_car": {
      "car_brand": "Hyundai",
      "brand_from": "Korea"
    },
    "car_brand": [
      "Subaru",
      "Hyundai"
    ],
    "countries": "Czech Republic",
    "colors": [
      {
        "hex": "#CD5C5C",
        "label": "IndianRed",
        "rgb": "205, 92, 92"
      },
      {
        "hex": "#F08080",
        "label": "LightCoral",
        "rgb": "240, 128, 128"
      }
    ]
  },
  {
    "user_id": 2,
    "first_name": "Gert",
    "last_name": "Fake",
    "gender": "male",
    "adore_car": {
      "car_brand": "Subaru",
      "brand_from": "Japan"
    },
    "car_brand": [
      "Subaru",
      "Dodge"
    ],
    "countries": "Colombia",
    "colors": [
      {
        "hex": "#E9967A",
        "label": "DarkSalmon",
        "rgb": "233, 150, 122"
      },
      {
        "hex": "#FFA07A",
        "label": "LightSalmon",
        "rgb": "255, 160, 122"
    


## Question 2 Cosine Similarity

Write `cosine_similarity(vec1, vec2) -> float` from scratch.

$$\text{cosine\_sim} = \frac{A \cdot B}{\|A\| \times \|B\|}$$

Expected output: `0.9926`

In [130]:
import math
from typing import List


def cosine_similarity(vec1: List[float], vec2: List[float]) -> float:
    dot = sum(a * b for a, b in zip(vec1, vec2))
    mag1 = math.sqrt(sum(a ** 2 for a in vec1))
    mag2 = math.sqrt(sum(b ** 2 for b in vec2))
    return round(dot / (mag1 * mag2), 4)


vec1 = [1, 2, 3]
vec2 = [2, 3, 4]
print(cosine_similarity(vec1, vec2))  # => 0.9926

0.9926


## Question 3 Sentiment Classification Pipeline

Build a mini ML pipeline:
1. **Preprocess**  lowercase, remove punctuation & stopwords, tokenize
2. **Feature Engineering**  TF-IDF
3. **Model**  Logistic Regression (train 80% / test 20%)
4. **Evaluate**  accuracy + confusion matrix

In [131]:
import requests
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

In [132]:
# load dataset from URL

url = "https://lief-assets-storage.sgp1.cdn.digitaloceanspaces.com/Test/dataset.txt"
dataset = requests.get(url).json()

texts = [item["text"] for item in dataset]
labels = [item["label"] for item in dataset]

print(f"Loaded {len(texts)} samples")

Loaded 1000 samples


In [133]:
# preprocess text: lowercase, remove punctuation, remove stopwords, tokenize

STOPWORDS = {"is", "the", "and", "a", "an", "of"}


def preprocess(text: str) -> str:
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = [t for t in text.split() if t not in STOPWORDS]
    return " ".join(tokens)


texts_clean = [preprocess(t) for t in texts]

for orig, clean in zip(texts[:3], texts_clean[:3]):
    print(f"{orig}  =>  {clean}")

I absolutely loved this movie, it was fantastic!  =>  i absolutely loved this movie it was fantastic
Terrible film. Waste of my time.  =>  terrible film waste my time
The story was touching and the characters were believable.  =>  story was touching characters were believable


In [134]:
# convert text to TF-IDF vectors

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts_clean)
y = labels

print(f"TF-IDF matrix shape: {X.shape}")

TF-IDF matrix shape: (1000, 45)


In [135]:
# split train 80% / test 20%

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")

Train: 800  |  Test: 200


In [123]:
# train logistic regression model

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

print("Model trained.")

In [124]:
# evaluate: accuracy and confusion matrix

y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"\nConfusion Matrix:")
print(cm)